# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the metadata, not as a subscriptable dict but as a single object
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset (including record sets, fields, columns, etc.) are referenced using their unique `@id` field.

In [ ]:
# Explore available record sets
record_sets = dataset.record_sets()
print("Available record sets and their @id:")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name', 'No name')}): {rs.get('description', '')}")

# List fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord set {rs['@id']} fields:")
    fields = rs.get('field', [])
    if not fields:
        print("  No fields found.")
    else:
        for field in fields:
            if isinstance(field, dict):  # When fields are returned as dict
                print(f"  {field['@id']} (name: {field.get('name', '')}, type: {field.get('dataType', '')})")
            else:
                print(f"  {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll use the record set `@id` for extraction. Adjust the code to select specific record set IDs based on the output above.

In [ ]:
# Identify record set IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
dataframes = {}

# Load each record set as a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display column names for the first available record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set '{first_record_set_id}':")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()
else:
    print("No records loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Fields should be referenced by their `@id`.

In [ ]:
# Choose a numeric field @id from the first record set
# You may need to adjust the field id based on the overview output.
record_set_id = first_record_set_id
df = dataframes[record_set_id]

# Attempt to find numeric fields
numeric_field_ids = [col for col in df.columns if 'score' in col.lower() or 'age' in col.lower() or 'GAD_7_score' in col or 'PHQ_9_score' in col]
if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
else:
    numeric_field = df.columns[0]  # Default fallback

# Filter for values greater than threshold
threshold = 10
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g., gender or education)
    group_field_candidates = [col for col in df.columns if ('gender' in col.lower() or 'education' in col.lower())]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print(f"Numeric field {numeric_field} not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field].dropna().hist(bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# If grouping field available, plot average scores
if 'group_field' in locals() and group_field in df.columns:
    avg_scores = df.groupby(group_field)[numeric_field].mean()
    avg_scores.plot(kind='bar', title=f"Average {numeric_field} by {group_field}", figsize=(8,4))
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset, referenced by its Croissant schema and loaded via `mlcroissant`, provides survey-based mental health indicators for Kilifi County, Kenya, with detailed demographic data.
- We examined available record sets and fields using their `@id`s and loaded tabular data into pandas DataFrames.
- Initial EDA revealed numeric fields relating to standardized mental health scores such as GAD-7 and PHQ-9, which were filtered and normalized for further analysis.
- Visualization of these scores highlights their distribution and possible demographic trends, informing potential research directions.

Further analysis could integrate machine learning models or deeper statistical analysis as appropriate for the dataset's content.